# 🎯 Notebook 12: Causal Inference from Observational Data

> Run with `jupyter nbconvert --to notebook --execute <this notebook>` (or open it in JupyterLab) to populate outputs. Every random step is seeded, so results are reproducible.

## Aims

Regression coefficients are causal only under strong assumptions. This notebook simulates
data in which the *true* causal effect is known and compares design-based estimators from
`utils.causal` against the naive comparison:

1. **Inverse probability weighting (IPW)** and **matching** under selection on observables
2. **Difference-in-differences (DiD)** under parallel trends
3. **Two-stage least squares (2SLS)** under an unmeasured confounder with a valid instrument
4. **Sharp regression discontinuity (RD)** with bandwidth sensitivity
5. **E-values** for sensitivity to unmeasured confounding

Because every dataset is simulated here with `numpy.random.default_rng`, we can report
bias directly.

In [ ]:
# Setup and imports
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.causal import (
    estimate_propensity, covariate_balance, ipw, nearest_neighbor_match, matching_att,
    difference_in_differences, two_stage_least_squares,
    regression_discontinuity, rd_bandwidth_sensitivity, e_value,
)
from utils.reporting import save_table
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = Path("../synthetic_data")
EXPORT_PLOTS = Path("../exports/plots/12_causal_inference")
EXPORT_TABLES = Path("../exports/tables/12_causal_inference")
EXPORT_PLOTS.mkdir(parents=True, exist_ok=True)
EXPORT_TABLES.mkdir(parents=True, exist_ok=True)

SEED = 2026

In [ ]:
comparison = []  # collects (design, estimator, truth, estimate, ci) for the final table

def record(design, estimator, truth, est, lo=np.nan, hi=np.nan):
    comparison.append({"design": design, "estimator": estimator, "true_effect": truth,
                       "estimate": est, "ci_low": lo, "ci_high": hi, "bias": est - truth})

## 1. Selection on observables: IPW and matching

**DGP.** Two covariates `x1`, `x2` raise both the probability of treatment and the outcome.
The true average treatment effect (ATE) is **2.0** for everyone, so ATE = ATT.

```
P(T = 1 | x) = logistic(0.8 x1 - 0.5 x2)
Y = 1 + 2.0 T + 1.5 x1 + 1.0 x2 + e,  e ~ N(0, 1)
```

In [ ]:
rng = np.random.default_rng(SEED)
n = 2000
x1, x2 = rng.normal(size=n), rng.normal(size=n)
p_treat = 1 / (1 + np.exp(-(0.8 * x1 - 0.5 * x2)))
T = rng.binomial(1, p_treat)
Y = 1 + 2.0 * T + 1.5 * x1 + 1.0 * x2 + rng.normal(size=n)
obs = pd.DataFrame({"x1": x1, "x2": x2, "treat": T, "Y": Y})
TRUE_ATE = 2.0

naive = obs.loc[obs.treat == 1, "Y"].mean() - obs.loc[obs.treat == 0, "Y"].mean()
print(f"Naive difference in means: {naive:.3f}  (truth = {TRUE_ATE})")
record("selection on observables", "naive difference", TRUE_ATE, naive)

In [ ]:
ate = ipw(obs, "Y", "treat", ["x1", "x2"], estimand="ATE", stabilized=True, n_boot=100, seed=SEED)
att = ipw(obs, "Y", "treat", ["x1", "x2"], estimand="ATT")
print(f"IPW ATE = {ate['estimate']:.3f}  [{ate['ci_low']:.3f}, {ate['ci_high']:.3f}]  "
      f"(bootstrap SE {ate['boot_se']:.3f}); effective n = {ate['effective_sample_size']:.0f}")
print(f"IPW ATT = {att['estimate']:.3f}  [{att['ci_low']:.3f}, {att['ci_high']:.3f}]")
record("selection on observables", "IPW (ATE, stabilised)", TRUE_ATE, ate["estimate"], ate["ci_low"], ate["ci_high"])
record("selection on observables", "IPW (ATT)", TRUE_ATE, att["estimate"], att["ci_low"], att["ci_high"])

In [ ]:
# Love plot: standardised mean differences before and after weighting
bal = pd.DataFrame({"before": ate["balance_before"]["smd"], "after IPW": ate["balance_after"]["smd"]})
fig, ax = plt.subplots(figsize=(6, 3))
y = np.arange(len(bal))
ax.scatter(bal["before"].abs(), y, label="before", color="firebrick")
ax.scatter(bal["after IPW"].abs(), y, label="after IPW", color="seagreen")
ax.axvline(0.1, ls="--", color="gray", label="0.1 threshold")
ax.set_yticks(y); ax.set_yticklabels(bal.index)
ax.set_xlabel("|standardised mean difference|")
ax.set_title("Covariate balance (love plot)")
ax.legend()
plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "love_plot_ipw.png")
plt.show()
bal

In [ ]:
ps = estimate_propensity(obs, "treat", ["x1", "x2"])
matched = nearest_neighbor_match(obs, "treat", ps, caliper=0.2, replace=False, seed=SEED)
m_att = matching_att(matched, "Y", "treat")
print(f"Matched {matched.attrs['n_matched']} of {matched.attrs['n_treated']} treated units")
print(f"Matching ATT = {m_att['estimate']:.3f}  [{m_att['ci_low']:.3f}, {m_att['ci_high']:.3f}]")
print(covariate_balance(matched, "treat", ["x1", "x2"]))
record("selection on observables", "1:1 caliper matching (ATT)", TRUE_ATE, m_att["estimate"], m_att["ci_low"], m_att["ci_high"])

## 2. Difference-in-differences

**DGP.** 60 units observed in two periods; half are treated in period 2. Treated units have a
higher baseline level (0.7) and every unit shares a common time trend (0.5), so both a
cross-sectional comparison and a before/after comparison are biased. The true effect is **3.0**.

In [ ]:
rng = np.random.default_rng(SEED + 1)
rows = []
for u in range(60):
    treated = int(u < 30)
    fe = rng.normal()
    for p in (0, 1):
        y = 1 + fe + 0.7 * treated + 0.5 * p + 3.0 * treated * p + rng.normal(scale=0.5)
        rows.append({"unit": u, "treated": treated, "post": p, "y": y})
panel = pd.DataFrame(rows)
TRUE_DID = 3.0

did = difference_in_differences(panel, "y", "treated", "post", cluster="unit")
print(f"DiD estimate = {did['estimate']:.3f}  [{did['ci_low']:.3f}, {did['ci_high']:.3f}], p = {did['p_value']:.2g}")
print(f"Manual 2x2 DiD  = {did['manual_did']:.3f}")
cross = panel.loc[panel.post == 1].groupby("treated")["y"].mean().diff().iloc[-1]
before_after = panel.loc[panel.treated == 1].groupby("post")["y"].mean().diff().iloc[-1]
print(f"Cross-sectional (post only): {cross:.3f}; before/after (treated only): {before_after:.3f}")
record("DiD", "cross-sectional post-period", TRUE_DID, cross)
record("DiD", "before/after treated", TRUE_DID, before_after)
record("DiD", "difference-in-differences", TRUE_DID, did["estimate"], did["ci_low"], did["ci_high"])

means = did["group_period_means"]
fig, ax = plt.subplots(figsize=(6, 3.5))
for g, lab in [(0, "control"), (1, "treated")]:
    ax.plot([0, 1], means.loc[g], marker="o", label=lab)
ax.plot([0, 1], [means.loc[1, 0], means.loc[1, 0] + (means.loc[0, 1] - means.loc[0, 0])], ls="--", color="gray", label="treated counterfactual")
ax.set_xticks([0, 1]); ax.set_xticklabels(["pre", "post"])
ax.set_ylabel("mean outcome"); ax.set_title("Difference-in-differences")
ax.legend(); plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "did_means.png")
plt.show()

## 3. Instrumental variables (2SLS)

**DGP.** An unobserved confounder `u` affects both `x` and `y`, so OLS is biased upward.
The instrument `z` shifts `x` but affects `y` only through `x`. The true effect is **1.0**.

In [ ]:
rng = np.random.default_rng(SEED + 2)
n = 3000
u = rng.normal(size=n)
z = rng.normal(size=n)
x = 0.8 * z + u + rng.normal(scale=0.5, size=n)
y = 1 + 1.0 * x + 2.0 * u + rng.normal(size=n)
iv_df = pd.DataFrame({"x": x, "y": y, "z": z})
TRUE_IV = 1.0

iv = two_stage_least_squares(iv_df, "y", "x", instruments=["z"])
print(f"OLS estimate  = {iv['ols_estimate']:.3f} (biased)")
print(f"2SLS estimate = {iv['estimate']:.3f}  [{iv['ci_low']:.3f}, {iv['ci_high']:.3f}]")
print(f"First-stage F = {iv['first_stage_f']:.1f}; weak instruments: {iv['weak_instruments']}")
record("unmeasured confounding + instrument", "OLS", TRUE_IV, iv["ols_estimate"])
record("unmeasured confounding + instrument", "2SLS", TRUE_IV, iv["estimate"], iv["ci_low"], iv["ci_high"])

## 4. Sharp regression discontinuity

**DGP.** Treatment is assigned when the running variable `r >= 0`. The outcome has a smooth
slope (0.5) plus a jump of **2.0** at the cutoff. We estimate the jump with a local linear
regression using a triangular kernel and check sensitivity to the bandwidth.

In [ ]:
rng = np.random.default_rng(SEED + 3)
n = 3000
r = rng.uniform(-1, 1, n)
y = 1 + 0.5 * r + 2.0 * (r >= 0) + 0.3 * r**2 + rng.normal(scale=0.3, size=n)
rd_df = pd.DataFrame({"r": r, "y": y})
TRUE_RD = 2.0

rd = regression_discontinuity(rd_df, "y", "r", cutoff=0.0, kernel="triangular", polynomial=1)
print(f"RD estimate = {rd['estimate']:.3f}  [{rd['ci_low']:.3f}, {rd['ci_high']:.3f}], "
      f"bandwidth = {rd['bandwidth']:.3f}, n = {rd['n_left']} + {rd['n_right']}")
record("regression discontinuity", "local linear (rule-of-thumb h)", TRUE_RD, rd["estimate"], rd["ci_low"], rd["ci_high"])

# Binned scatter with the fitted local-linear lines inside the bandwidth
h = rd["bandwidth"]
bins = pd.cut(rd_df.r, np.linspace(-1, 1, 41))
binned = rd_df.groupby(bins, observed=True).agg(r=("r", "mean"), y=("y", "mean"))
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(binned.r, binned.y, s=18, color="gray", label="binned means")
params = rd["result"].params
grid_l = np.linspace(-h, 0, 50); grid_r = np.linspace(0, h, 50)
ax.plot(grid_l, params["const"] + params["x1"] * grid_l, color="C0", lw=2, label="fit (left)")
ax.plot(grid_r, params["const"] + params["D"] + (params["x1"] + params["D_x1"]) * grid_r, color="C3", lw=2, label="fit (right)")
ax.axvline(0, ls="--", color="k")
ax.set_xlabel("running variable r"); ax.set_ylabel("y")
ax.set_title("Sharp RD: jump at the cutoff")
ax.legend(); plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "rd_fit.png")
plt.show()

In [ ]:
sens = rd_bandwidth_sensitivity(rd_df, "y", "r", bandwidths=[0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0], kernel="triangular")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.errorbar(sens.bandwidth, sens.estimate, yerr=[sens.estimate - sens.ci_low, sens.ci_high - sens.estimate], fmt="o-", capsize=3)
ax.axhline(TRUE_RD, color="red", ls="--", label="true jump")
ax.set_xlabel("bandwidth"); ax.set_ylabel("RD estimate")
ax.set_title("Bandwidth sensitivity (local linear, triangular kernel)")
ax.legend(); plt.tight_layout()
plt.savefig(EXPORT_PLOTS / "rd_bandwidth_sensitivity.png")
plt.show()
sens

Wide bandwidths bring in the curvature (`0.3 r^2`) that a linear fit cannot absorb, so the
estimate drifts; narrow bandwidths are noisier. Reporting a sensitivity curve is standard practice.

## 5. Sensitivity to unmeasured confounding: E-values

The E-value is the minimum strength of association (on the risk-ratio scale) that an
unmeasured confounder would need with *both* treatment and outcome to explain away the
observed estimate. For a standardised mean difference `d`, `RR ≈ exp(0.91 d)`.

In [ ]:
sd_y = obs["Y"].std()
d_ipw = ate["estimate"] / sd_y
ev = e_value(d_ipw, ate["ci_low"] / sd_y, ate["ci_high"] / sd_y, scale="d")
print(f"IPW effect as SMD: d = {d_ipw:.3f}")
print(f"E-value for the point estimate: {ev['e_value']:.2f}; for the CI bound: {ev['e_value_ci']:.2f}")

ev_grid = pd.DataFrame([{"d": d, "RR_equiv": e_value(d, scale="d")["rr_equivalent"], "e_value": e_value(d, scale="d")["e_value"]} for d in [0.1, 0.2, 0.5, 0.8, 1.0, 1.5]])
ev_grid

## 6. Summary: every estimator against the truth

In [ ]:
final = pd.DataFrame(comparison)
final["covered"] = (final.ci_low <= final.true_effect) & (final.true_effect <= final.ci_high)
save_table(final, EXPORT_TABLES / "causal_estimators_vs_truth", formats=("csv", "md"), index=False)
final

## References

- Rosenbaum, P. R., & Rubin, D. B. (1983). The central role of the propensity score in observational studies for causal effects. *Biometrika*, 70(1), 41–55.
- Hirano, K., Imbens, G. W., & Ridder, G. (2003). Efficient estimation of average treatment effects using the estimated propensity score. *Econometrica*, 71(4), 1161–1189.
- Austin, P. C. (2011). An introduction to propensity score methods for reducing the effects of confounding in observational studies. *Multivariate Behavioral Research*, 46(3), 399–424.
- Card, D., & Krueger, A. B. (1994). Minimum wages and employment. *American Economic Review*, 84(4), 772–793.
- Bertrand, M., Duflo, E., & Mullainathan, S. (2004). How much should we trust differences-in-differences estimates? *QJE*, 119(1), 249–275.
- Staiger, D., & Stock, J. H. (1997). Instrumental variables regression with weak instruments. *Econometrica*, 65(3), 557–586.
- Imbens, G. W., & Lemieux, T. (2008). Regression discontinuity designs: A guide to practice. *Journal of Econometrics*, 142(2), 615–635.
- Lee, D. S., & Lemieux, T. (2010). Regression discontinuity designs in economics. *Journal of Economic Literature*, 48(2), 281–355.
- VanderWeele, T. J., & Ding, P. (2017). Sensitivity analysis in observational research: Introducing the E-value. *Annals of Internal Medicine*, 167(4), 268–274.
- Imbens, G. W., & Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences*. Cambridge University Press.